# Alert Rules Motor Calibration

This notebook documents the complementary calculations used to calibrate the Module 2 decision engine from the Module 1 historical analysis. The goal is to leave a reproducible trail for:

- base rain trigger and sensitive-zone override
- the trade-off between `forecast 1h` and `forecast 3h`
- recommended `EARNINGS` target
- event duration for alert de-duplication
- proximity between zones to suggest secondary zones

The notebook uses the same cleaned dataset as the main `DataAnalysis` module.

## Setup

It is recommended to run this notebook with the `DataAnalysis` environment, because `pandas`, `pyarrow`, and `jupyter` are already available there.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
pd.set_option('display.max_rows', 200)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'DataAnalysis').exists() and (candidate / 'EarlyAlertsAPI').exists():
            return candidate
    raise FileNotFoundError('Project root could not be found.')


def root_relative(path: Path) -> str:
    relative = path.relative_to(PROJECT_ROOT)
    relative_text = relative.as_posix()
    return relative_text or '.'


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ANALYSIS = PROJECT_ROOT / 'DataAnalysis'
EARLY_ALERTS = PROJECT_ROOT / 'EarlyAlertsAPI'

RAW_PATH = DATA_ANALYSIS / 'outputs' / 'cleaned' / 'raw_data_clean.parquet'
ZONE_INFO_PATH = DATA_ANALYSIS / 'outputs' / 'cleaned' / 'zone_info_clean.parquet'
SUMMARY_PATH = DATA_ANALYSIS / 'outputs' / 'reports' / 'resumen_hallazgos_modulo1_1_pagina.md'
P2_BUCKETS_PATH = DATA_ANALYSIS / 'outputs' / 'tables' / 'p2_rain_bucket_ratio.csv'
P2_HOUR_LIFT_PATH = DATA_ANALYSIS / 'outputs' / 'tables' / 'p2_hour_control_ratio_lift.csv'
P3_SENSITIVITY_PATH = DATA_ANALYSIS / 'outputs' / 'tables' / 'p3_zone_precipitation_sensitivity.csv'
P5_INTERACTION_PATH = DATA_ANALYSIS / 'outputs' / 'tables' / 'p5_earnings_rain_ratio_interaction.csv'

print('PROJECT_ROOT =', root_relative(PROJECT_ROOT))
print('RAW_PATH =', root_relative(RAW_PATH))
print('ZONE_INFO_PATH =', root_relative(ZONE_INFO_PATH))


PROJECT_ROOT = .
RAW_PATH = DataAnalysis/outputs/cleaned/raw_data_clean.parquet
ZONE_INFO_PATH = DataAnalysis/outputs/cleaned/zone_info_clean.parquet


## Load Data and Module 1 References

In [2]:
df = pd.read_parquet(RAW_PATH).copy()
zone_info = pd.read_parquet(ZONE_INFO_PATH).copy()
p2_buckets = pd.read_csv(P2_BUCKETS_PATH)
p2_hour_lift = pd.read_csv(P2_HOUR_LIFT_PATH)
p3_sensitivity = pd.read_csv(P3_SENSITIVITY_PATH)
p5_interaction = pd.read_csv(P5_INTERACTION_PATH)

df['RATIO'] = df['ORDERS'] / df['CONNECTED_RT'].replace(0, np.nan)
df['DATETIME'] = pd.to_datetime(df['DATE']) + pd.to_timedelta(df['HOUR'], unit='h')
df['IS_PEAK'] = df['HOUR'].isin([12, 13, 14, 19, 20, 21])
df['EARNINGS_Q'] = pd.qcut(df['EARNINGS'], q=4, labels=['Q1_low', 'Q2', 'Q3', 'Q4_high'])
df = df.sort_values(['ZONE', 'DATETIME']).reset_index(drop=True)

print('Rows:', len(df))
print('Zones:', df['ZONE'].nunique())
print('Date range:', df['DATE'].min(), '->', df['DATE'].max())

display(p2_buckets)
display(p3_sensitivity.sort_values('ratio_lift', ascending=False).head(6))
display(p5_interaction)


Rows: 10080
Zones: 14
Date range: 2024-03-01 00:00:00 -> 2024-03-30 00:00:00


,RAIN_BUCKET,avg_ratio,median_ratio,avg_connected_rt,avg_orders,pct_hours_saturated,n,delta_orders_vs_no_rain,delta_connected_rt_vs_no_rain
0,no_rain,0.774945,0.777778,8.940406,8.867498,3.677092,9464,0.000000,0.000000
1,light,1.060386,0.875000,8.936975,11.424370,17.226891,238,2.556872,-0.003431
2,moderate,1.700395,1.500000,13.097403,21.116883,31.168831,154,12.249385,4.156997
3,heavy,1.769480,1.428571,15.183036,23.504464,33.928571,224,14.636966,6.242630


,ZONE,spearman_precip_ratio,p_value,avg_ratio_no_rain,avg_ratio_moderate_plus,ratio_lift,orders_lift,connected_rt_lift,response_gap_orders_minus_rt,n_observations,n_moderate_plus
0,Santiago,0.2730,0.0,0.618,2.439,1.821,6.691,0.924,5.767,720,27
1,Carretera Nacional,0.2719,0.0,0.642,2.126,1.484,9.044,1.980,7.063,720,27
2,MTY_Apodaca_Huinalá,0.2235,0.0,0.614,1.675,1.061,6.609,2.485,4.124,720,27
3,Santa Catarina,0.2420,0.0,0.877,1.868,0.991,11.690,3.669,8.021,720,27
4,San Pedro,0.2282,0.0,0.790,1.666,0.876,20.093,8.018,12.075,720,27
5,San Nicolás,0.2134,0.0,0.789,1.639,0.850,18.094,8.134,9.960,720,27


,EARNINGS_Q,RAIN_BUCKET,avg_ratio,pct_hours_saturated,n
0,Q1_low,heavy,3.291270,83.333333,6
1,Q1_low,light,1.701186,40.000000,10
2,Q1_low,moderate,2.115219,60.000000,5
3,Q1_low,no_rain,0.816332,6.832298,2576
4,Q2,heavy,2.352063,62.500000,48
5,Q2,light,1.082960,22.500000,80
6,Q2,moderate,1.906301,41.176471,34
7,Q2,no_rain,0.758070,5.408859,2348
8,Q3,heavy,2.083921,59.183673,49
9,Q3,light,0.997369,10.256410,78


## 1. Rain Trigger Threshold: Citywide vs Peak Hours

Here we review why `2.0 mm/hr` remains the base trigger and `5.0 mm/hr` is used only for escalation.

In [3]:
def summarize_thresholds(frame: pd.DataFrame, thresholds=(0.1, 1.0, 2.0, 3.0, 5.0)) -> pd.DataFrame:
    rows = []
    for th in thresholds:
        sub = frame[frame['PRECIPITATION_MM'] >= th]
        rows.append({
            'threshold_mm': th,
            'n': int(len(sub)),
            'avg_ratio': round(sub['RATIO'].mean(), 3),
            'pct_saturated': round((sub['RATIO'] > 1.8).mean() * 100, 1),
            'avg_earnings': round(sub['EARNINGS'].mean(), 2),
        })
    return pd.DataFrame(rows)


overall_thresholds = summarize_thresholds(df)
peak_thresholds = summarize_thresholds(df[df['IS_PEAK']])

print('Thresholds - all hours')
display(overall_thresholds)

print('Thresholds - peak hours only')
display(peak_thresholds)


Thresholds - all hours


,threshold_mm,n,avg_ratio,pct_saturated,avg_earnings
0,0.1,616,1.478,26.8,63.82
1,1.0,448,1.713,33.7,66.64
2,2.0,378,1.741,32.8,67.00
3,3.0,336,1.684,29.2,66.77
4,5.0,224,1.769,33.9,68.98


Thresholds - peak hours only


,threshold_mm,n,avg_ratio,pct_saturated,avg_earnings
0,0.1,308,1.988,49.4,67.04
1,1.0,266,2.037,51.9,68.73
2,2.0,238,1.992,47.1,70.11
3,3.0,196,1.948,43.9,70.39
4,5.0,154,1.989,45.5,71.27


## 2. Sensitive-Zone Override in Peak Hours

This checks that some zones already move into high risk with `1.0-1.9 mm/hr` if rain lands in lunch/dinner windows.

In [4]:
zone_name_map = {
    'MTY_Apodaca_Huinalá': 'MTY_Apodaca_Huinala',
    'San Nicolás': 'San Nicolas',
}

peak = df[df['IS_PEAK']].copy()
peak['ZONE_DOC'] = peak['ZONE'].replace(zone_name_map)

rows = []
for zone, grp in peak.groupby('ZONE_DOC'):
    base = grp[grp['PRECIPITATION_MM'] < 0.1]
    for th in [1.0, 2.0, 5.0]:
        sub = grp[grp['PRECIPITATION_MM'] >= th]
        rows.append({
            'ZONE': zone,
            'threshold_mm': th,
            'n': int(len(sub)),
            'avg_ratio': round(sub['RATIO'].mean(), 3) if len(sub) else np.nan,
            'pct_saturated': round((sub['RATIO'] > 1.8).mean() * 100, 1) if len(sub) else np.nan,
            'base_no_rain_peak_ratio': round(base['RATIO'].mean(), 3) if len(base) else np.nan,
        })

zone_thresholds = pd.DataFrame(rows)
display(zone_thresholds.sort_values(['threshold_mm', 'pct_saturated'], ascending=[True, False]))

sensitive_floor = (
    zone_thresholds[zone_thresholds['threshold_mm'] == 1.0]
    .sort_values('avg_ratio', ascending=False)
    [['ZONE', 'avg_ratio', 'pct_saturated']]
)
print('Suggested floors for the sensitive override (threshold = 1.0 mm/hr in peak hours):')
display(sensitive_floor.head(6))


,ZONE,threshold_mm,n,avg_ratio,pct_saturated,base_no_rain_peak_ratio
3,Carretera Nacional,1.0,19,2.407,68.4,1.386
39,Santiago,1.0,19,2.700,57.9,1.368
0,Apodaca Centro,1.0,19,1.996,52.6,1.423
6,Centro,1.0,19,1.925,52.6,1.364
12,Escobedo,1.0,19,1.937,52.6,1.410
15,Independencia,1.0,19,1.937,52.6,1.371
24,MTY_Guadalupe,1.0,19,1.927,52.6,1.415
27,Mitras Centro,1.0,19,1.831,52.6,1.401
33,San Pedro,1.0,19,1.981,52.6,1.386
36,Santa Catarina,1.0,19,2.171,52.6,1.388


Suggested floors for the sensitive override (threshold = 1.0 mm/hr in peak hours):


,ZONE,avg_ratio,pct_saturated
39,Santiago,2.700,57.9
3,Carretera Nacional,2.407,68.4
36,Santa Catarina,2.171,52.6
21,MTY_Apodaca_Huinala,2.120,47.4
0,Apodaca Centro,1.996,52.6
33,San Pedro,1.981,52.6


## 3. Lead-Time Trade-off: `t+1` vs `t+3`

We measure the relationship between rain observed at `t` and future `RATIO` at `t+k` to see how much signal remains at `+1h`, `+2h`, and `+3h`.

In [5]:
def lagged_effect(frame: pd.DataFrame, ks=(0, 1, 2, 3), rain_threshold=2.0) -> pd.DataFrame:
    rows = []
    temp = frame[['ZONE', 'DATETIME', 'PRECIPITATION_MM']].copy()
    for k in ks:
        future = frame[['ZONE', 'DATETIME', 'RATIO']].copy()
        future['DATETIME'] = future['DATETIME'] - pd.to_timedelta(k, unit='h')
        merged = temp.merge(future, on=['ZONE', 'DATETIME'], how='inner')
        rain = merged[merged['PRECIPITATION_MM'] >= rain_threshold]
        dry = merged[merged['PRECIPITATION_MM'] < 0.1]
        rows.append({
            'k_hours': k,
            'rain_n': int(len(rain)),
            'avg_ratio_if_precip_ge_2': round(rain['RATIO'].mean(), 3),
            'pct_sat_if_precip_ge_2': round((rain['RATIO'] > 1.8).mean() * 100, 1),
            'avg_ratio_if_dry': round(dry['RATIO'].mean(), 3),
            'pct_sat_if_dry': round((dry['RATIO'] > 1.8).mean() * 100, 1),
        })
    return pd.DataFrame(rows)


lag_all_hours = lagged_effect(df)
lag_peak_hours = lagged_effect(df[df['IS_PEAK']])

print('Lagged effect - all hours')
display(lag_all_hours)

print('Lagged effect - peak hours')
display(lag_peak_hours)


Lagged effect - all hours


,k_hours,rain_n,avg_ratio_if_precip_ge_2,pct_sat_if_precip_ge_2,avg_ratio_if_dry,pct_sat_if_dry
0,0,378,1.741,32.8,0.775,3.7
1,1,378,1.448,22.0,0.791,4.3
2,2,378,1.149,13.5,0.801,4.5
3,3,378,0.948,10.3,0.813,4.9


Lagged effect - peak hours


,k_hours,rain_n,avg_ratio_if_precip_ge_2,pct_sat_if_precip_ge_2,avg_ratio_if_dry,pct_sat_if_dry
0,0,238,1.992,47.1,1.388,15.7
1,1,140,1.870,41.4,1.416,17.4
2,2,56,1.692,28.6,1.446,18.9
3,3,0,NaN,NaN,NaN,NaN


## 4. Earnings Target

The current earnings level under peak rainy conditions is compared with the historical high quartile to define a single operational target.

In [6]:
earnings_summary = df['EARNINGS'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).round(2)
print('Overall EARNINGS distribution')
display(earnings_summary.to_frame(name='EARNINGS'))

peak_moderate_plus = df[(df['IS_PEAK']) & (df['PRECIPITATION_MM'] >= 2.0)].copy()
q4_peak_moderate_plus = peak_moderate_plus[peak_moderate_plus['EARNINGS_Q'] == 'Q4_high']

target_summary = pd.DataFrame([
    {
        'metric': 'current_median_peak_moderate_plus',
        'value': round(peak_moderate_plus['EARNINGS'].median(), 2),
    },
    {
        'metric': 'current_mean_peak_moderate_plus',
        'value': round(peak_moderate_plus['EARNINGS'].mean(), 2),
    },
    {
        'metric': 'q4_median_peak_moderate_plus',
        'value': round(q4_peak_moderate_plus['EARNINGS'].median(), 2),
    },
    {
        'metric': 'q4_mean_peak_moderate_plus',
        'value': round(q4_peak_moderate_plus['EARNINGS'].mean(), 2),
    },
    {
        'metric': 'delta_median_to_q4',
        'value': round(q4_peak_moderate_plus['EARNINGS'].median() - peak_moderate_plus['EARNINGS'].median(), 2),
    },
    {
        'metric': 'baseline_city_median',
        'value': round(df['EARNINGS'].median(), 2),
    },
    {
        'metric': 'delta_city_median_to_80',
        'value': round(80.0 - df['EARNINGS'].median(), 2),
    },
])

display(target_summary)

zone_q4 = (
    peak_moderate_plus.groupby('ZONE')['EARNINGS'].median().rename('current_median_earnings')
    .to_frame()
    .join(q4_peak_moderate_plus.groupby('ZONE')['EARNINGS'].median().rename('q4_median_earnings'))
)
zone_q4['delta_to_q4_median'] = zone_q4['q4_median_earnings'] - zone_q4['current_median_earnings']
display(zone_q4.sort_values('delta_to_q4_median', ascending=False))


Overall EARNINGS distribution


,EARNINGS
count,10080.00
mean,57.70
std,8.29
min,49.00
10%,50.40
25%,52.40
50%,55.60
75%,58.70
90%,71.90
95%,75.90


,metric,value
0,current_median_peak_moderate_plus,70.50
1,current_mean_peak_moderate_plus,70.11
2,q4_median_peak_moderate_plus,80.30
3,q4_mean_peak_moderate_plus,79.52
4,delta_median_to_q4,9.80
5,baseline_city_median,55.60
6,delta_city_median_to_80,24.40


,current_median_earnings,q4_median_earnings,delta_to_q4_median
ZONE,,,
Carretera Nacional,70.0,81.70,11.70
San Nicolás,72.3,83.40,11.10
Cumbres Poniente,70.2,81.10,10.90
Independencia,70.0,80.75,10.75
Santiago,70.4,80.95,10.55
Santa Catarina,70.2,80.50,10.30
Apodaca Centro,71.7,81.90,10.20
San Pedro,70.7,80.20,9.50
Centro,71.9,80.20,8.30


## 5. Event Duration and Secondary Zones

The median duration of `moderate+` events is estimated for alert cooldown, and nearby zones are computed from centroids.

In [7]:
city = df[['DATE', 'HOUR', 'PRECIPITATION_MM']].drop_duplicates().sort_values(['DATE', 'HOUR']).copy()
city['rain_event'] = (city['PRECIPITATION_MM'] >= 2.0).astype(int)
city['start'] = city['rain_event'].eq(1) & city['rain_event'].shift(fill_value=0).eq(0)
city['event_id'] = city['start'].cumsum()
event_lengths = city[city['rain_event'] == 1].groupby('event_id').size()

event_summary = pd.DataFrame([
    {'metric': 'moderate_plus_event_count', 'value': int(len(event_lengths))},
    {'metric': 'moderate_plus_event_median_hours', 'value': float(event_lengths.median())},
    {'metric': 'moderate_plus_event_max_hours', 'value': int(event_lengths.max())},
    {'metric': 'moderate_plus_event_lengths', 'value': str(event_lengths.tolist())},
])
display(event_summary)

lat_col = 'LATITUDE_CENTER'
lon_col = 'LONGITUDE_CENTER'
zone_info['ZONE_DOC'] = zone_info['ZONE'].replace(zone_name_map)

nearest_rows = []
for _, a in zone_info.iterrows():
    distances = []
    for _, b in zone_info.iterrows():
        if a['ZONE_DOC'] == b['ZONE_DOC']:
            continue
        d = math.sqrt((a[lat_col] - b[lat_col]) ** 2 + (a[lon_col] - b[lon_col]) ** 2)
        distances.append((d, b['ZONE_DOC']))
    distances.sort()
    nearest_rows.append({
        'ZONE': a['ZONE_DOC'],
        'nearest_1': distances[0][1],
        'nearest_2': distances[1][1],
        'nearest_3': distances[2][1],
    })

nearest_zones = pd.DataFrame(nearest_rows).sort_values('ZONE')
display(nearest_zones)


,metric,value
0,moderate_plus_event_count,7
1,moderate_plus_event_median_hours,4.0
2,moderate_plus_event_max_hours,6
3,moderate_plus_event_lengths,"[5, 3, 4, 4, 3, 2, 6]"


,ZONE,nearest_1,nearest_2,nearest_3
2,Apodaca Centro,La Fe,San Nicolas,MTY_Apodaca_Huinala
4,Carretera Nacional,MTY_Guadalupe,Independencia,Santiago
0,Centro,Independencia,Mitras Centro,San Pedro
9,Cumbres Poniente,Mitras Centro,Santa Catarina,Escobedo
3,Escobedo,San Nicolas,Mitras Centro,Apodaca Centro
12,Independencia,Centro,San Pedro,Mitras Centro
10,La Fe,San Nicolas,Centro,Independencia
5,MTY_Apodaca_Huinala,MTY_Guadalupe,Apodaca Centro,La Fe
11,MTY_Guadalupe,MTY_Apodaca_Huinala,La Fe,Carretera Nacional
1,Mitras Centro,Centro,Independencia,San Pedro


## 6. Proposed Final Rules

This table summarizes the result carried into the engine documentation.

In [8]:
recommended_rules = pd.DataFrame([
    {'component': 'trigger_base', 'rule': 'forecast t+1 >= 2.0 mm/hr', 'support': 'P2 + threshold analysis'},
    {'component': 'trigger_sensitive_peak', 'rule': 'forecast t+1 >= 1.0 mm/hr in Santiago, Carretera Nacional, Santa Catarina, and MTY_Apodaca_Huinala', 'support': 'P3 + peak zone analysis'},
    {'component': 'severity_escalation', 'rule': '>= 5.0 mm/hr escalates to critical if the case is already notifiable', 'support': 'threshold analysis'},
    {'component': 'lead_time', 'rule': '1h active; 3h watchlist only', 'support': 'lagged effect analysis'},
    {'component': 'earnings_target', 'rule': 'raise to 80 MXN/order', 'support': 'P5 + earnings target analysis'},
    {'component': 'cooldown', 'rule': '4h per zone-event', 'support': 'event duration analysis'},
    {'component': 'event_close', 'rule': 'close after 2 consecutive hours below 0.1 mm/hr or below the active trigger', 'support': 'anti-dup design based on event duration'},
])
display(recommended_rules)


,component,rule,support
0,trigger_base,forecast t+1 >= 2.0 mm/hr,P2 + threshold analysis
1,trigger_sensitive_peak,"forecast t+1 >= 1.0 mm/hr in Santiago, Carrete...",P3 + peak zone analysis
2,severity_escalation,>= 5.0 mm/hr escalates to critical if the case...,threshold analysis
3,lead_time,1h active; 3h watchlist only,lagged effect analysis
4,earnings_target,raise to 80 MXN/order,P5 + earnings target analysis
5,cooldown,4h per zone-event,event duration analysis
6,event_close,close after 2 consecutive hours below 0.1 mm/h...,anti-dup design based on event duration
